In [ ]:
"""
Temporary filling-point cost assignment (Nigeria LPG chain, EPSG:3857)

This notebook applies a temporary simplification:
- all points in the `filling` layer receive `cost_kg = 0.6`

Important
---------
- this is an interim assumption only;
- filling-point cost will be computed with a more robust method later.
- the original merged GeoPackage is NOT modified;
  a copied GeoPackage is created and updated for downstream steps.

Behavior
--------
1) verifies that the source GeoPackage exists,
2) creates/overwrites a copied GeoPackage for this temporary scenario,
3) verifies that `filling` exists in the copied file,
4) sets `cost_kg = 0.6` for all rows in `filling` in the copied file only.
"""

from __future__ import annotations

import shutil
import sqlite3
from pathlib import Path

import fiona

DATA_DIR = Path("dataset_big")
SOURCE_GPKG_PATH = DATA_DIR / "full_lpg_chain_nig_3857.gpkg"
OUTPUT_GPKG_PATH = DATA_DIR / "chain_with_cost.gpkg"

FILLING_LAYER = "filling"
COST_COL = "cost_kg"
TEMP_FILLING_COST = 0.6


def main() -> None:
    print("[1/4] Checking source GeoPackage...")
    if not SOURCE_GPKG_PATH.exists():
        raise FileNotFoundError(f"Source file not found: {SOURCE_GPKG_PATH}")

    print("[2/4] Creating copied GeoPackage for temporary-cost scenario...")
    if OUTPUT_GPKG_PATH.exists():
        OUTPUT_GPKG_PATH.unlink()
    shutil.copy2(SOURCE_GPKG_PATH, OUTPUT_GPKG_PATH)

    print("[3/4] Validating layers in copied GeoPackage...")
    layers = fiona.listlayers(str(OUTPUT_GPKG_PATH))
    if not layers:
        raise RuntimeError(f"No layers found in copied file: {OUTPUT_GPKG_PATH}")
    if FILLING_LAYER not in layers:
        raise KeyError(f"Required layer '{FILLING_LAYER}' not found in {OUTPUT_GPKG_PATH}")

    print("[4/4] Updating only filling points in copied file...")
    with sqlite3.connect(str(OUTPUT_GPKG_PATH)) as conn:
        cur = conn.cursor()

        cols = cur.execute(f'PRAGMA table_info("{FILLING_LAYER}")').fetchall()
        col_names = {c[1] for c in cols}
        if COST_COL not in col_names:
            cur.execute(f'ALTER TABLE "{FILLING_LAYER}" ADD COLUMN "{COST_COL}" REAL')

        cur.execute(
            f'UPDATE "{FILLING_LAYER}" SET "{COST_COL}" = ?',
            (float(TEMP_FILLING_COST),),
        )
        updated = cur.rowcount
        conn.commit()

    print(f"Done. Updated {updated:,} rows in '{FILLING_LAYER}' in copied file only.")
    print(f"Source unchanged: {SOURCE_GPKG_PATH}")
    print(f"Output file:      {OUTPUT_GPKG_PATH}")


main()

In [ ]:
"""
Assign one filling reference to each reseller using shortest-path travel time
on the motorized friction graph (Dijkstra), then write outputs to the copied GPKG.

Outputs:
- resell.filling_reference     (ID of assigned filling point; id_res&fil convention)
- resell.filling_ref_time_min  (shortest-path travel time in minutes)
- raster: filling_preferred_by_resell_id.tif
- raster: filling_preferred_by_resell_time_min.tif
"""

from __future__ import annotations

import sqlite3
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, dijkstra
from scipy.spatial import cKDTree

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"

MOTO_FRICTION_RASTER = DATA_DIR / "friction_moto.tif"
OUT_RASTER_FILL_ID = DATA_DIR / "filling_preferred_by_resell_id.tif"
OUT_RASTER_FILL_TIME = DATA_DIR / "filling_preferred_by_resell_time_min.tif"

ID_COL = "id_res&fil"
OUT_REF_COL = "filling_reference"
OUT_TIME_COL = "filling_ref_time_min"

CELL_SIZE_METERS = 1000.0
USE_8_NEIGHBORS = False
MIN_CANDIDATES = 4
PRIMARY_SEARCH_RADIUS_KM = 60
MAX_SEARCH_RADIUS_KM = 120
INITIAL_LIMIT_FACTOR = 10.0
FINAL_LIMIT_FACTOR = 16.0
LIMIT_MARGIN_MIN = 30.0
UNASSIGNED_TIME_MIN = 7000.0
NODATA_INT = -1
NODATA_FLOAT = -9999.0
PROGRESS_EVERY = 200

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def _read_friction(path: Path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        nodata = src.nodata
        profile = src.profile.copy()
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    arr = np.where(arr > 0, arr, np.nan).astype(np.float32)
    return arr, profile


def _map_points_to_grid(gdf: gpd.GeoDataFrame, transform, width: int, height: int):
    rows, cols = rasterio.transform.rowcol(transform, gdf.geometry.x.values, gdf.geometry.y.values)
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)
    inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    return rows, cols, inside


def _candidate_idx_adaptive(r: int, c: int, tree: cKDTree, coords: np.ndarray) -> np.ndarray:
    idx = np.array([], dtype=np.int32)

    found_primary = tree.query_ball_point([r, c], r=PRIMARY_SEARCH_RADIUS_KM, p=2)
    if len(found_primary) > 0:
        idx = np.asarray(found_primary, dtype=np.int32)

    if idx.size < MIN_CANDIDATES:
        found_max = tree.query_ball_point([r, c], r=MAX_SEARCH_RADIUS_KM, p=2)
        if len(found_max) > 0:
            idx = np.unique(np.concatenate([idx, np.asarray(found_max, dtype=np.int32)]))

    if idx.size < MIN_CANDIDATES:
        k = min(max(MIN_CANDIDATES, 8), len(coords))
        _, nn = tree.query([r, c], k=k)
        idx = np.unique(np.concatenate([idx, np.atleast_1d(nn).astype(np.int32)]))

    return idx


def _pick_min_time(dist_row: np.ndarray, cand_idx: np.ndarray, fill_nodes: np.ndarray, fill_ids: np.ndarray):
    cand_nodes = fill_nodes[cand_idx]
    t = dist_row[cand_nodes]
    if not np.isfinite(t).any():
        return NODATA_INT, np.nan
    j = int(np.nanargmin(t))
    return int(fill_ids[cand_idx[j]]), float(t[j])


def _run_assign(
    src_node: int,
    r: int,
    c: int,
    graph: csr_matrix,
    friction_min: float,
    base_idx: np.ndarray,
    fill_rows: np.ndarray,
    fill_cols: np.ndarray,
    fill_nodes: np.ndarray,
    fill_ids: np.ndarray,
    src_component: int,
    fill_component: np.ndarray,
    ):
    cand = np.unique(base_idx)
    if cand.size > 0:
        cand = cand[fill_component[cand] == src_component]

    if cand.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN, "no_candidate_in_component"

    lb = np.hypot(fill_rows[cand] - r, fill_cols[cand] - c) * CELL_SIZE_METERS * max(friction_min, 1e-9)
    limit = float(np.nanmax(lb) * INITIAL_LIMIT_FACTOR + LIMIT_MARGIN_MIN) if lb.size > 0 else np.inf
    dist_row = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit,
    )[0]

    fid, tmin = _pick_min_time(dist_row, cand, fill_nodes, fill_ids)
    if fid >= 0 and np.isfinite(tmin):
        return fid, float(tmin), "base_ok"

    cand_global = np.where(fill_component == src_component)[0].astype(np.int32)
    if cand_global.size == 0:
        return NODATA_INT, UNASSIGNED_TIME_MIN, "no_global_in_component"

    lb2 = np.hypot(fill_rows[cand_global] - r, fill_cols[cand_global] - c) * CELL_SIZE_METERS * max(friction_min, 1e-9)
    limit2 = float(np.nanmax(lb2) * FINAL_LIMIT_FACTOR + LIMIT_MARGIN_MIN) if lb2.size > 0 else np.inf
    dist_row2 = dijkstra(
        csgraph=graph,
        directed=True,
        indices=[int(src_node)],
        unweighted=False,
        limit=limit2,
    )[0]

    fid2, tmin2 = _pick_min_time(dist_row2, cand_global, fill_nodes, fill_ids)
    if fid2 >= 0 and np.isfinite(tmin2):
        return fid2, float(tmin2), "fallback_ok"

    return NODATA_INT, UNASSIGNED_TIME_MIN, "fallback_unassigned"


def _write_int_raster(path: Path, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="int32", count=1, nodata=NODATA_INT, compress="lzw")
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(array.astype(np.int32), 1)


def _write_float_raster(path: Path, array: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(dtype="float32", count=1, nodata=NODATA_FLOAT, compress="lzw")
    out = np.where(np.isfinite(array), array, NODATA_FLOAT).astype(np.float32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)


# ---------------------------------------------------------------------
# RUN
# ---------------------------------------------------------------------
t0 = time.time()
print("[1/7] Reading friction raster and building valid mask...")
friction, ref_profile = _read_friction(MOTO_FRICTION_RASTER)
height, width = friction.shape
transform = ref_profile["transform"]
crs = ref_profile["crs"]

valid = np.isfinite(friction)
if not np.any(valid):
    raise RuntimeError("No valid cells in motorized friction raster.")
friction_min = float(np.nanpercentile(friction[valid], 5))
print(f"Grid: {width} x {height} | valid cells: {int(valid.sum()):,}")

print("[2/7] Loading reseller and filling points from copied GeoPackage...")
resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty:
    raise RuntimeError("Resell layer is empty.")
if filling.empty:
    raise RuntimeError("Filling layer is empty.")

for name, gdf in ((RESELL_LAYER, resell), (FILLING_LAYER, filling)):
    if ID_COL not in gdf.columns:
        raise KeyError(f"Missing '{ID_COL}' in layer '{name}'.")
    if gdf.geometry.isna().all():
        raise RuntimeError(f"All geometries are null in layer '{name}'.")

if resell.crs != crs:
    resell = resell.to_crs(crs)
if filling.crs != crs:
    filling = filling.to_crs(crs)

resell = resell[resell.geometry.notna() & resell.geometry.geom_type.isin(["Point"])].copy()
filling = filling[filling.geometry.notna() & filling.geometry.geom_type.isin(["Point"])].copy()

rid_resell = pd.to_numeric(resell[ID_COL], errors="coerce")
rid_fill = pd.to_numeric(filling[ID_COL], errors="coerce")
if rid_resell.isna().any() or (rid_resell <= 0).any() or (not rid_resell.is_unique):
    raise ValueError(f"Layer '{RESELL_LAYER}' must have unique positive numeric '{ID_COL}'.")
if rid_fill.isna().any() or (rid_fill <= 0).any() or (not rid_fill.is_unique):
    raise ValueError(f"Layer '{FILLING_LAYER}' must have unique positive numeric '{ID_COL}'.")

print("[3/7] Mapping points to friction grid...")
res_rows, res_cols, in_res = _map_points_to_grid(resell, transform, width, height)
fill_rows, fill_cols, in_fill = _map_points_to_grid(filling, transform, width, height)

resell = resell.loc[in_res].copy()
filling = filling.loc[in_fill].copy()
res_rows = res_rows[in_res]
res_cols = res_cols[in_res]
fill_rows = fill_rows[in_fill]
fill_cols = fill_cols[in_fill]
rid_resell = rid_resell.loc[in_res].astype(np.int32).to_numpy()
rid_fill = rid_fill.loc[in_fill].astype(np.int32).to_numpy()

if len(resell) == 0 or len(filling) == 0:
    raise RuntimeError("No points left inside raster extent after mapping.")

print(f"Resellers on grid: {len(resell):,}")
print(f"Fillings on grid : {len(filling):,}")

print("[4/7] Building shortest-path graph on friction_moto...")
node_id = -np.ones((height, width), dtype=np.int32)
vr, vc = np.where(valid)
node_id[vr, vc] = np.arange(len(vr), dtype=np.int32)
n_nodes = len(vr)

if USE_8_NEIGHBORS:
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]
else:
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]

edge_i = []
edge_j = []
edge_c = []
diag_factor = np.sqrt(2.0)

for r, c in zip(vr, vc):
    n0 = node_id[r, c]
    f0 = friction[r, c]
    for dr, dc in neighbors:
        rr = r + dr
        cc = c + dc
        if rr < 0 or rr >= height or cc < 0 or cc >= width:
            continue
        n1 = node_id[rr, cc]
        if n1 < 0:
            continue
        f1 = friction[rr, cc]
        if not np.isfinite(f1):
            continue

        step_m = CELL_SIZE_METERS
        if dr != 0 and dc != 0:
            step_m *= diag_factor

        cost = 0.5 * (f0 + f1) * step_m
        if cost <= 0:
            continue

        edge_i.append(int(n0))
        edge_j.append(int(n1))
        edge_c.append(float(cost))

graph = csr_matrix((edge_c, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
n_comp, comp = connected_components(csgraph=graph, directed=False, return_labels=True)
print(f"Graph nodes: {n_nodes:,} | edges: {len(edge_i):,} | components: {n_comp:,}")

res_nodes = node_id[res_rows, res_cols]
fill_nodes = node_id[fill_rows, fill_cols]

ok_res = res_nodes >= 0
ok_fill = fill_nodes >= 0
res_nodes = res_nodes[ok_res]
res_rows = res_rows[ok_res]
res_cols = res_cols[ok_res]
rid_resell = rid_resell[ok_res]

fill_nodes = fill_nodes[ok_fill]
fill_rows = fill_rows[ok_fill]
fill_cols = fill_cols[ok_fill]
rid_fill = rid_fill[ok_fill]

if len(res_nodes) == 0 or len(fill_nodes) == 0:
    raise RuntimeError("No reseller/filling mapped to valid graph nodes.")

fill_comp = comp[fill_nodes]
fill_tree = cKDTree(np.column_stack([fill_rows, fill_cols]).astype(np.float64))

print("[5/7] Assigning nearest filling per reseller (Dijkstra + adaptive candidates)...")
n_res = len(res_nodes)
ref_id = np.full(n_res, NODATA_INT, dtype=np.int32)
ref_time = np.full(n_res, UNASSIGNED_TIME_MIN, dtype=np.float32)
reasons = {}
loop_t0 = time.time()

for k, (src_node, r, c) in enumerate(zip(res_nodes, res_rows, res_cols), start=1):
    base_idx = _candidate_idx_adaptive(int(r), int(c), fill_tree, fill_tree.data)
    src_comp = int(comp[int(src_node)])
    fid, tmin, reason = _run_assign(
        src_node=int(src_node),
        r=int(r),
        c=int(c),
        graph=graph,
        friction_min=friction_min,
        base_idx=base_idx,
        fill_rows=fill_rows,
        fill_cols=fill_cols,
        fill_nodes=fill_nodes,
        fill_ids=rid_fill,
        src_component=src_comp,
        fill_component=fill_comp,
    )
    ref_id[k - 1] = int(fid)
    ref_time[k - 1] = float(tmin)
    reasons[reason] = int(reasons.get(reason, 0)) + 1

    if (k % PROGRESS_EVERY == 0) or (k == n_res):
        elapsed = time.time() - loop_t0
        speed = k / max(elapsed, 1e-9)
        rem = (n_res - k) / max(speed, 1e-9)
        print(f"  {k:,}/{n_res:,} ({100.0*k/n_res:.1f}%) | {speed:.1f} res/s | ETA ~{rem/60:.1f} min")

assigned_ok = (ref_id >= 0) & np.isfinite(ref_time) & (ref_time < UNASSIGNED_TIME_MIN)
print(f"Assigned resellers: {int(assigned_ok.sum()):,}/{n_res:,}")
print("Reason counts:", reasons)

print("[6/7] Updating reseller table in copied GeoPackage...")
updates = pd.DataFrame({
    ID_COL: rid_resell,
    OUT_REF_COL: ref_id,
    OUT_TIME_COL: ref_time,
})

with sqlite3.connect(str(WORK_GPKG)) as conn:
    cur = conn.cursor()
    cols = [c[1] for c in cur.execute(f'PRAGMA table_info("{RESELL_LAYER}")').fetchall()]
    if OUT_REF_COL not in cols:
        cur.execute(f'ALTER TABLE "{RESELL_LAYER}" ADD COLUMN "{OUT_REF_COL}" INTEGER')
    if OUT_TIME_COL not in cols:
        cur.execute(f'ALTER TABLE "{RESELL_LAYER}" ADD COLUMN "{OUT_TIME_COL}" REAL')

    cur.executemany(
        f'UPDATE "{RESELL_LAYER}" SET "{OUT_REF_COL}" = ?, "{OUT_TIME_COL}" = ? WHERE "{ID_COL}" = ?',
        [
            (int(a), float(t), int(rid))
            for a, t, rid in zip(
                updates[OUT_REF_COL].to_numpy(),
                updates[OUT_TIME_COL].to_numpy(),
                updates[ID_COL].to_numpy(),
            )
        ],
    )
    conn.commit()

print("[7/7] Writing visual rasters for QA...")
rast_id = np.full((height, width), NODATA_INT, dtype=np.int32)
rast_time = np.full((height, width), np.nan, dtype=np.float32)

for r, c, a, t in zip(res_rows, res_cols, ref_id, ref_time):
    rast_id[r, c] = int(a)
    rast_time[r, c] = float(t)

_write_int_raster(OUT_RASTER_FILL_ID, rast_id, ref_profile)
_write_float_raster(OUT_RASTER_FILL_TIME, rast_time, ref_profile)

print(f"Updated layer: {WORK_GPKG} | {RESELL_LAYER}")
print(f"- Column added/updated: {OUT_REF_COL}")
print(f"- Column added/updated: {OUT_TIME_COL}")
print(f"Raster ID   : {OUT_RASTER_FILL_ID}")
print(f"Raster time : {OUT_RASTER_FILL_TIME}")
print(f"Done in {(time.time() - t0)/60:.1f} min")

In [ ]:
"""
Update reseller cost_kg using assigned filling reference.

Formula agreed:
    cost_kg_resell = cost_kg_filling_reference + filling_ref_time_min / 1000
"""

from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
COST_COL = "cost_kg"

print("[1/4] Loading required fields from copied GeoPackage...")
with sqlite3.connect(str(WORK_GPKG)) as conn:
    q_resell = (
        f'SELECT "{ID_COL}" AS rid, "{FILL_REF_COL}" AS fref, "{FILL_TIME_COL}" AS tmin '
        f'FROM "{RESELL_LAYER}"'
    )
    q_fill = (
        f'SELECT "{ID_COL}" AS fid, "{COST_COL}" AS fcost '
        f'FROM "{FILLING_LAYER}"'
    )
    resell_df = pd.read_sql_query(q_resell, conn)
    filling_df = pd.read_sql_query(q_fill, conn)

print("[2/4] Validating data and computing new reseller cost_kg...")
resell_df["rid"] = pd.to_numeric(resell_df["rid"], errors="coerce")
resell_df["fref"] = pd.to_numeric(resell_df["fref"], errors="coerce")
resell_df["tmin"] = pd.to_numeric(resell_df["tmin"], errors="coerce")
filling_df["fid"] = pd.to_numeric(filling_df["fid"], errors="coerce")
filling_df["fcost"] = pd.to_numeric(filling_df["fcost"], errors="coerce")

if resell_df["rid"].isna().any() or (resell_df["rid"] <= 0).any() or (not resell_df["rid"].is_unique):
    raise ValueError(f"Layer '{RESELL_LAYER}' has invalid '{ID_COL}' values.")
if filling_df["fid"].isna().any() or (filling_df["fid"] <= 0).any() or (not filling_df["fid"].is_unique):
    raise ValueError(f"Layer '{FILLING_LAYER}' has invalid '{ID_COL}' values.")

map_filling_cost = dict(zip(filling_df["fid"].astype(np.int64), filling_df["fcost"].astype(float)))
resell_df["fill_cost"] = resell_df["fref"].map(map_filling_cost)

valid = (
    resell_df["fref"].notna()
    & resell_df["tmin"].notna()
    & resell_df["fill_cost"].notna()
    & (resell_df["fref"] > 0)
    & np.isfinite(resell_df["tmin"])
    & np.isfinite(resell_df["fill_cost"])
    & (resell_df["tmin"] >= 0)
    & (resell_df["tmin"] < 7000)
    & (resell_df["fill_cost"] >= 0)
 )

resell_df["new_cost"] = np.where(
    valid,
    resell_df["fill_cost"] + (resell_df["tmin"] / 1000.0),
    np.nan,
 )

n_valid = int(valid.sum())
n_total = int(len(resell_df))
if n_valid == 0:
    raise RuntimeError("No valid reseller rows found for cost update.")

print("[3/4] Writing updated cost_kg to reseller layer...")
updates = resell_df.loc[valid, ["rid", "new_cost"]].copy()

with sqlite3.connect(str(WORK_GPKG)) as conn:
    cur = conn.cursor()
    cols = [c[1] for c in cur.execute(f'PRAGMA table_info("{RESELL_LAYER}")').fetchall()]
    if COST_COL not in cols:
        cur.execute(f'ALTER TABLE "{RESELL_LAYER}" ADD COLUMN "{COST_COL}" REAL')

    cur.executemany(
        f'UPDATE "{RESELL_LAYER}" SET "{COST_COL}" = ? WHERE "{ID_COL}" = ?',
        [
            (float(c), int(r))
            for c, r in zip(updates["new_cost"].to_numpy(), updates["rid"].to_numpy())
        ],
    )
    conn.commit()

print("[4/4] Done.")
print(f"Updated rows: {n_valid:,}/{n_total:,} in layer '{RESELL_LAYER}'")
print(f"Output file: {WORK_GPKG}")
print("Formula used: cost_kg_resell = cost_kg_filling + filling_ref_time_min / 1000")

In [ ]:
"""
Final QA stats for 4.4 outputs.

Checks:
1) filling points have updated cost_kg (not legacy 999) and sensible values
2) resell points have a valid filling_reference and sensible travel stats
3) resell points have updated cost_kg (not legacy 999) and sensible values
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
COST_COL = "cost_kg"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"

print("[1/4] Loading layers...")
resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")

for layer_name, gdf in ((RESELL_LAYER, resell), (FILLING_LAYER, filling)):
    for c in [ID_COL, COST_COL]:
        if c not in gdf.columns:
            raise KeyError(f"Missing column '{c}' in layer '{layer_name}'")
if FILL_REF_COL not in resell.columns:
    raise KeyError(f"Missing column '{FILL_REF_COL}' in layer '{RESELL_LAYER}'")
if FILL_TIME_COL not in resell.columns:
    raise KeyError(f"Missing column '{FILL_TIME_COL}' in layer '{RESELL_LAYER}'")

fill_id = pd.to_numeric(filling[ID_COL], errors="coerce")
fill_cost = pd.to_numeric(filling[COST_COL], errors="coerce")
res_id = pd.to_numeric(resell[ID_COL], errors="coerce")
res_ref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
res_tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
res_cost = pd.to_numeric(resell[COST_COL], errors="coerce")

print("[2/4] QA on filling cost_kg...")
n_fill = len(filling)
fill_valid = np.isfinite(fill_cost)
fill_not_999 = fill_valid & (~np.isclose(fill_cost, 999.0))
fill_nonneg = fill_valid & (fill_cost >= 0)

print("=== FILLING COST QA ===")
print(f"Filling points total         : {n_fill:,}")
print(f"Finite cost_kg               : {int(fill_valid.sum()):,}")
print(f"cost_kg != 999               : {int(fill_not_999.sum()):,}")
print(f"cost_kg == 999 (legacy)      : {int((fill_valid & np.isclose(fill_cost, 999.0)).sum()):,}")
print(f"cost_kg < 0                  : {int((fill_valid & (fill_cost < 0)).sum()):,}")
if fill_valid.any():
    print(
        "cost_kg min/median/mean/max : "
        f"{float(np.nanmin(fill_cost)):.4f} / {float(np.nanmedian(fill_cost)):.4f} / "
        f"{float(np.nanmean(fill_cost)):.4f} / {float(np.nanmax(fill_cost)):.4f}"
    )

print("\n[3/4] QA on reseller filling reference and travel stats...")
n_res = len(resell)
fill_id_set = set(fill_id[np.isfinite(fill_id)].astype(np.int64).tolist())
ref_finite = np.isfinite(res_ref)
ref_positive = ref_finite & (res_ref > 0)
ref_exists = ref_positive & pd.Series(res_ref.astype("Int64")).isin(fill_id_set).to_numpy()

time_valid = np.isfinite(res_tmin) & (res_tmin >= 0) & (res_tmin < 7000)
ref_and_time_valid = ref_exists & time_valid

print("=== RESELLER REFERENCE QA ===")
print(f"Resell points total                    : {n_res:,}")
print(f"Resell with finite filling_reference   : {int(ref_finite.sum()):,}")
print(f"Resell with existing filling_reference : {int(ref_exists.sum()):,}")
print(f"Resell with valid reference + time     : {int(ref_and_time_valid.sum()):,}")
print(f"Missing/invalid reference              : {int((~ref_exists).sum()):,}")
if ref_and_time_valid.any():
    t = res_tmin[ref_and_time_valid].to_numpy(dtype=float)
    print(
        "Travel time min/median/mean/p95/max (min): "
        f"{float(np.nanmin(t)):.2f} / {float(np.nanmedian(t)):.2f} / {float(np.nanmean(t)):.2f} / "
        f"{float(np.nanpercentile(t, 95)):.2f} / {float(np.nanmax(t)):.2f}"
    )

# Optional geometric sanity check in km (straight-line)
if (resell.crs is not None) and (filling.crs is not None) and (str(resell.crs) == str(filling.crs)):
    fill_geom = filling[[ID_COL, "geometry"]].copy()
    fill_geom[ID_COL] = pd.to_numeric(fill_geom[ID_COL], errors="coerce")
    merged = resell[[ID_COL, FILL_REF_COL, "geometry"]].copy()
    merged[ID_COL] = pd.to_numeric(merged[ID_COL], errors="coerce")
    merged[FILL_REF_COL] = pd.to_numeric(merged[FILL_REF_COL], errors="coerce")
    merged = merged.merge(fill_geom, left_on=FILL_REF_COL, right_on=ID_COL, how="left", suffixes=("_res", "_fill"))
    good_geom = merged["geometry_res"].notna() & merged["geometry_fill"].notna()
    if good_geom.any():
        dist_km = merged.loc[good_geom, "geometry_res"].distance(merged.loc[good_geom, "geometry_fill"]).astype(float) / 1000.0
        print(
            "Straight-line distance min/median/mean/p95/max (km): "
            f"{float(np.nanmin(dist_km)):.2f} / {float(np.nanmedian(dist_km)):.2f} / {float(np.nanmean(dist_km)):.2f} / "
            f"{float(np.nanpercentile(dist_km, 95)):.2f} / {float(np.nanmax(dist_km)):.2f}"
        )

print("\n[4/4] QA on reseller cost_kg...")
res_cost_valid = np.isfinite(res_cost)
res_cost_not_999 = res_cost_valid & (~np.isclose(res_cost, 999.0))
res_cost_nonneg = res_cost_valid & (res_cost >= 0)
res_cost_updated_and_linked = res_cost_not_999 & ref_and_time_valid

print("=== RESELLER COST QA ===")
print(f"Finite cost_kg                        : {int(res_cost_valid.sum()):,}/{n_res:,}")
print(f"cost_kg != 999                        : {int(res_cost_not_999.sum()):,}/{n_res:,}")
print(f"cost_kg updated with valid reference  : {int(res_cost_updated_and_linked.sum()):,}/{n_res:,}")
print(f"cost_kg == 999 (legacy)               : {int((res_cost_valid & np.isclose(res_cost, 999.0)).sum()):,}")
print(f"cost_kg < 0                           : {int((res_cost_valid & (res_cost < 0)).sum()):,}")
if res_cost_valid.any():
    print(
        "cost_kg min/median/mean/p95/max      : "
        f"{float(np.nanmin(res_cost)):.4f} / {float(np.nanmedian(res_cost)):.4f} / {float(np.nanmean(res_cost)):.4f} / "
        f"{float(np.nanpercentile(res_cost, 95)):.4f} / {float(np.nanmax(res_cost)):.4f}"
    )

print("\nQA completed.")